# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), stratified 5-fold CV on training rows only, choice by mean CV ROC AUC, holdout scored once.

In [ ]:
import hashlib, json
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open("data.csv","rb").read()).hexdigest()=="25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4"
assert hashlib.sha256(open("holdout.csv","rb").read()).hexdigest()=="083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf"
df=pd.read_csv("data.csv")
hold_idx=pd.read_csv("holdout.csv")["row"].to_numpy()
assert len(df)==7000 and len(hold_idx)==1400 and len(set(hold_idx))==1400
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
print(len(train_idx),len(hold_idx))

## Features
`device_serial` is excluded (identifier, per protocol). The other four columns (channel, device age, price, reported daily use at registration) are all recorded at or before sale, so they are available at prediction time.

In [ ]:
target="claim_within_year"
features=["channel","device_age_years","price_eur","daily_hours"]
cat=["channel"]; num=[f for f in features if f not in cat]
X=df[features]; y=df[target].to_numpy()
Xtr,ytr=X.iloc[train_idx],y[train_idx]
Xho,yho=X.iloc[hold_idx],y[hold_idx]
print(ytr.mean(), yho.mean())

## Candidates (named before any scoring)
Preprocessing lives inside each pipeline so it is fitted within CV folds.

In [ ]:
def pre(scale):
    return ColumnTransformer([
        ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler() if scale else "passthrough")]),num),
        ("cat",OneHotEncoder(handle_unknown="ignore"),cat)])
candidates={
 "baseline_prior":Pipeline([("pre",pre(False)),("m",DummyClassifier(strategy="prior"))]),
 "logreg_C1":Pipeline([("pre",pre(True)),("m",LogisticRegression(C=1.0,max_iter=1000))]),
 "logreg_C0.1":Pipeline([("pre",pre(True)),("m",LogisticRegression(C=0.1,max_iter=1000))]),
 "rf_depth6":Pipeline([("pre",pre(False)),("m",RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 "hgb_small":Pipeline([("pre",pre(False)),("m",HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows

In [ ]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=0)
folds=[(tr,va) for tr,va in skf.split(Xtr,ytr)]
cv_scores={}
for name,p in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(p).fit(Xtr.iloc[tr],ytr[tr])
        s.append(float(roc_auc_score(ytr[va],m.predict_proba(Xtr.iloc[va])[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame({k:{"mean":np.mean(v),"std":np.std(v)} for k,v in cv_scores.items()}).T.sort_values("mean",ascending=False)
cv

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print("chosen:",chosen)

## Holdout (scored once)

In [ ]:
final=clone(candidates[chosen]).fit(Xtr,ytr)
holdout_auc=float(roc_auc_score(yho,final.predict_proba(Xho)[:,1]))
print(f"Holdout ROC AUC for {chosen}: {holdout_auc:.4f}")

In [ ]:
# 0-based row positions in data.csv
evidence={
 "features":features,
 "candidates":list(candidates),
 "cv_folds":[[[int(train_idx[i]) for i in tr],[int(train_idx[i]) for i in va]] for tr,va in folds],
 "cv_scores":cv_scores,
 "chosen":chosen,
 "holdout_rows":[int(i) for i in hold_idx],
 "holdout_scores":{chosen:holdout_auc},
}
json.dump(evidence,open("evidence.json","w"))